In [ ]:
####################################
# Packages and solvers
####################################

!pip install --quiet ecos
#!pip install --quiet pyscipopt
#!pip install --quiet mosek       # NOTE: requires a licence file stored at /root/mosek/mosek.lic
import cvxpy as cp
import numpy as np
#import pyscipopt

In [ ]:
####################################
# Auxiliary functions
####################################

# Function that builds a set of outbound and inbound sailing legs for each route
def build_legs(routes):
  legs = {}

  N_route_ports = { key:np.size(routes[key]) for key in routes.keys() }
  for s in routes.keys():
    legs[s] = set()

    for k in range(N_route_ports[s]-1):
      i = routes[s][k]; j = routes[s][k+1]
      legs[s].add((i, j)); legs[s].add((j, i))

  return legs

# Function that builds a set of active origin-destination cargo flow arcs for each leg in each route
def build_leg_flow_arcs(routes):
  leg_flow_arcs = {}

  N_route_ports = { key:np.size(routes[key]) for key in routes.keys() }
  for s in np.arange(len(routes)):  # change to routes.keys()
    route_arcs = {}
    for k in range(N_route_ports[s]-1):
      arcs = []
      i = routes[s][k]; j = routes[s][k+1]
      # Loop from the initial departure port to the departure port of the current leg
      for k_o in range(0, k+1):
        # Loop from the current departure port to the final arrival port
        for k_d in range(k+1, N_route_ports[s]):
          i_p = routes[s][k_o]; j_p = routes[s][k_d]
          arcs.append((i_p, j_p))
      route_arcs[(i,j)] = arcs
    leg_flow_arcs[s] = route_arcs

  return leg_flow_arcs

# Function that builds a set of origin-destination cargo flow arcs for the transportation network
def build_flow_arcs(leg_demand_arcs):
  flow_arcs = {}
  for s in leg_demand_arcs.keys():
    flow_arcs[s] = set()
    for leg in leg_demand_arcs[s].keys():
      for (i, j) in leg_demand_arcs[s][leg]:
        flow_arcs[s].add((i, j)); flow_arcs[s].add((j, i))

  return flow_arcs

In [ ]:
####################################
# Log-convex problem implementation
####################################

def build_and_solve_logcvx(vessel_params, cost_params, network_params, solver='ECOS'):

  #
  # Data
  #

  beta, eta_el, gamma_batt, k_A, N_sup, P_aux_unit, A_sup_unit, beta_KB, eps_KG, sigma_y_max, k_safe, \
  rho_vol, rho_mass, rho_st, p_sup, p_min, h_sup, h_roro, h_batt_min, l_CB, V_int_hat, k_vol, N_crew_const, N_crew_var,  \
  = map(vessel_params.get, ['beta', 'eta_el', 'gamma_batt', 'k_A', 'N_sup', 'P_aux_unit', 'A_sup_unit',
                            'beta_KB', 'eps_KG', 'sigma_y_max', 'k_safe', 'rho_vol', 'rho_mass', 'rho_st',
                            'p_sup', 'p_min', 'h_sup', 'h_roro', 'h_batt_min', 'l_CB', 'V_int_hat', 'k_vol',
                            'N_crew_const', 'N_crew_var',
                            ])

  N_batt, C_st_ann, C_batt_ann, C_crew_ann, C_el_ann, C_port_ann, C_ship_ann \
  = map(cost_params.get, ['N_batt', 'C_st_ann', 'C_batt_ann','C_crew_ann', 'C_el_ann', 'C_port_ann', 'C_ship_ann'])

  U_min, L_od, q_dem, routes, t_avail, N_trip, N_ship_max, t_cargo, U_bias, P_cha_max, q_min \
  = map(network_params.get, ['U_min', 'L_od', 'q_dem', 'routes', 't_avail',
                             'N_trip', 'N_ship_max', 't_cargo', 'U_bias', 'P_cha_max', 'q_min'])


  #
  # Indexing sets
  #

  # Ports
  N_route_ports = { key:np.size(routes[key]) for key in routes.keys() }
  N_ports = max([max(v) for v in routes.values()])+1
  ports = np.arange(N_ports)

  # Vessels
  N_services = len(routes)
  services = np.arange(N_services)

  # Cargoes
  cargoes = np.arange(2)

  # Sailing legs
  leg_demand_arcs = build_leg_flow_arcs(routes)
  flow_arcs = build_flow_arcs(leg_demand_arcs)
  legs = build_legs(routes)

  # Demand origin-destination pairs and sailing legs
  active_od = set()

  for s in services:
    active_od = active_od.union(flow_arcs[s])

  #
  # Decision variables
  #

  f = {}        # cargo quantity shipped, 1000 pax or 1000 lane-m
  v = {}        # speed, km/h
  t_port = {}   # port turnaround time, h

  for s in services:
    for (i, j) in flow_arcs[s]:
      for c in cargoes:
        f[(i, j, s, c)] = cp.Variable(name=f"f_{i}_{j}_{s}_{c}", pos=True)

    for (i, j) in legs[s]:
      t_port[(i, j, s)] = cp.Variable(name=f"t_port_{i}_{j}_{s}", pos=True)

  N_roundtrip = cp.Variable(N_services, name="N_roundtrip", pos=True) # service frequency, including num of ships
  N_vessel = cp.Variable(N_services, name="N_vessel", pos=True)       # num of ships in a service

  v = cp.Variable(N_services, name="v", pos=True)
  r_Cf = cp.Variable(N_services, name=f"r_Cf", pos=True)
  r_Cr_std = cp.Variable(N_services, name=f"r_Cr_std", pos=True)

  L_sup = cp.Variable(N_services, name="L_sup", pos=True)             # superstructure length, m
  L = cp.Variable(N_services, name="L", pos=True)                     # hull length, m
  B = cp.Variable(N_services, name="B", pos=True)                     # hull waterline breadth, m
  T = cp.Variable(N_services, name="T", pos=True)                     # draft, m
  D = cp.Variable(N_services, name="D", pos=True)                     # hull depth (keel to topmost continuous deck), m
  p_td = cp.Variable(N_services, name="p_td", pos=True)               # deck plate thickness, m
  h_batt = cp.Variable(N_services, name="h_b", pos=True)              # battery deck height, m
  w_batt = cp.Variable((3, N_services), name="w_batt", pos=True)      # battery space width, m
  l_batt = cp.Variable((3, N_services), name="l_batt", pos=True)      # battery space length, m
  hp_batt = cp.Variable(N_services, name="hp_b", pos=True)            # battery space deck floor elevation from keel, m
  lp_batt = cp.Variable((3, N_services), name="lp_batt", pos=True)    # battery space front wall distance from origin (bow), m
  hp_roro = cp.Variable(N_services, name="hp_roro", pos=True)         # first roro deck elevation from keel, m
  E_batt = cp.Variable(N_services, name="E_batt", pos=True)           # battery capacity, MJ
  r_g = cp.Variable((4, N_services), name="r_g", pos=True)            # auxiliary variable for wetted area numerical integration
  r_g2 = cp.Variable((4, N_services), name="r_g2", pos=True)          # auxiliary variable for hull girder steel area area numerical integration

  #
  # Derived quantities
  #

  # Resistance
  k_BT = [(B[s]/T[s])**0.2748 for s in services]                      # beam–draft ratio factor
  k_LB = [(L[s]/B[s])**-0.5747 for s in services]                     # length–beam ratio factor
  k_Pr = 0.7**-0.1418                                                 # propeller factor
  k_rud = 2**-0.1258                                                  # rudder factor
  k_L  = [1.8319*(L[s]**-0.1237) for s in services]                   # length factor

  # Hull
  C_B = (0.5 - np.log(beta+1)/(2*beta) + beta/(3+3*beta))             # block coefficient
  nabla = [C_B*L[s]*B[s]*T[s] for s in services]                      # displacement, m^3
  h_KG = [0.7*D[s] for s in services]                                 # vertical coordinate of hull mass center, m
  V_int = [L[s]*B[s]*D[s]*k_vol + B[s]*L_sup[s]*(N_sup*h_sup +\
           h_roro) for s in services]                                 # internal volume, m^3
  V_GT = [V_int[s]*(0.2 + 0.02*np.log10(V_int_hat)*cp.power(V_int[s]/V_int_hat,\
          1/(np.log10(V_int_hat)*np.log(10)) )) for s in services]    # gross tonnage
  B_deck = [B[s]*cp.power(D[s]/T[s], 1/beta) for s in services]       # hull width at girder top deck, m
  A_wet = [2*L[s]*k_A*cp.sum([[1, 3, 3, 1][i]*cp.power(r_g[i, s], 0.5) \
                              for i in range(4) ]) for s in services] # hull wetted area, m^2
  A_girder = [2*L[s]*k_A*cp.sum([[1, 3, 3, 1][i]*cp.power(r_g2[i, s], 0.5) \
              for i in range(4) ]) for s in services]                 # external hull steel area, m^2
  f_cap = {0: [L[s]*B[s]/6000 for s in services],                    # cargo capacity, CHECK
       1: [N_sup*B[s]*L_sup[s]/7500 for s in services]}
  M_max = [9.81*C_B*cp.power(L[s], 2)*B[s]*T[s]/(2*(np.pi**2)*1000)\
           for s in services]                                         # max bending moment, MNm
  p_sp = [B[s]*p_td[s]/(2*D[s]) for s in services]                    # side plating thickness, m

  # Energy
  P_aux = [P_aux_unit*N_sup*B[s]*L_sup[s] for s in services]          # hotel power, kW

  # Masses
  W_side = [rho_st*p_sp[s]*A_girder[s] for s in services]             # external plating mass, t
  W_bottom = [rho_st*p_td[s]*L[s]*B[s]/2 for s in services]           # bottom plating mass, t
  W_deck = [rho_st*p_td[s]*2*5*B[s]*L[s]/6 for s in services]         # 2x roro deck mass, t
  W_tt = [rho_st*p_td[s]*2*B[s]*L[s]*((hp_batt[s]/T[s])**(1/beta))/3 \
          for s in services]                                          # tank top (battery space) deck plate mass, t
  W_tbh0 = [2*rho_st*p_sup*(beta/(beta+1))*hp_roro[s]*B[s]*((hp_roro[s]/T[s])**(1/beta))*((2*lp_batt[0, s]/L[s])**(0.5)) \
            for s in services]                                        # battery space transverse bulkheads (battery space 0) mass, t
  W_tbh1 = [2*rho_st*p_sup*(beta/(beta+1))*hp_roro[s]*B[s]*((hp_roro[s]/T[s])**(1/beta))*((2*lp_batt[1, s]/L[s])**(0.5)) \
            for s in services]                                        # battery space transverse bulkheads (battery space 1) mass, t
  W_tbh2 = [2*rho_st*p_sup*(beta/(beta+1))*hp_roro[s]*B[s]*((hp_roro[s]/T[s])**(1/beta))*((2*lp_batt[2, s]/L[s])**(0.5)) \
            for s in services]                                        # battery space transverse bulkheads (battery space 2) mass, t
  W_bwall = [rho_st*p_sup*h_batt[s]*(l_batt[0, s] + 2*l_batt[1, s] + 2*l_batt[2, s]) \
             for s in services]                                       # battery space wall mass, t
  W_bh = [rho_st*2*p_sp[s]*0.7*D[s]*L[s] for s in services]           # longitudinal bulkhead mass, t
  W_ramp = [rho_st*p_sup*B[s]*h_roro for s in services]               # stern roro ramp

  W_sup = [rho_st*p_sup*( N_sup*(2*L_sup[s]*h_sup + L_sup[s]*B[s] + 2*B[s]*h_sup) + 2*L_sup[s]*h_roro + \
           L_sup[s]*B[s] + B[s]*h_roro ) for s in services]           # superstructure plate mass, t
  W_plate = [W_side[s] + W_bottom[s] + W_bwall[s] + W_tbh0[s] + W_tbh1[s] + W_tbh2[s] + W_tt[s] + W_deck[s] + \
             W_ramp[s] + W_bh[s] + W_sup[s] for s in services]        # total plate mass
  W_fit = [0.2*(2*5*B[s]*L[s]/6 + N_sup*(B[s]*L_sup[s])) \
           for s in services]                                         # outfitting mass, t
  W_batt = [E_batt[s]*rho_mass/rho_vol for s in services]             # battery mass, t
  W_roro = [36*1000*f_cap[0][s]/23 for s in services]                 # cargo mass, t (23 m long vehicle that weighs 36 tons)
  W_sum = [1.1*W_plate[s] + W_fit[s] + W_batt[s] + W_roro[s] for s in services]

  R_fric = [0.5*75*cp.power(r_Cf[s], -2)*A_wet[s]*((v[s]/3.6)**2)/1000 for s in services]
  Fr = [(v[s]/3.6)*((L[s]*9.81)**-0.5) for s in services]
  R_res = [0.5*r_Cr_std[s]*k_L[s]*k_BT[s]*k_LB[s]*k_Pr*k_rud*((v[s]/3.6)**2)*B[s]*T[s]/10 for s in services]

  # Leg-specific quantities
  E_leg = {}                                                          # leg energy consumption, MJ (kN*km)
  for s in services:
    for (i, j) in legs[s]:
      E_leg[(i, j, s)] = (R_fric[s] + R_res[s])*L_od[i, j]/eta_el + (L_od[i,j]/v[s])*3.6*P_aux[s]

  L_od_norm = L_od/np.max(L_od)                                       # normalized sailing distance

  # Battery
  k_batt = N_batt**-0.7

  #
  # Network constraints
  #

  constr = [ ]

  # Restrict N_roundtrip and N_vessel to a set of valid integer-valued frequencies
  for s in services:
    integer_N_roundtrip = cp.FiniteSet(N_roundtrip[s], N_trip)
    constr += [integer_N_roundtrip]
    integer_N_vessel = cp.FiniteSet(N_vessel[s], np.arange(1, N_ship_max+1))
    constr += [integer_N_vessel]

  # Vessel capacity
  for s in services:
    for leg in leg_demand_arcs[s].keys(): # iterate legs (outbound) and impose a constr for each
      for c in cargoes:
        constr += [ cp.sum([f[(i, j, s, c)] for (i, j) in leg_demand_arcs[s][leg]]) <= f_cap[c][s]*N_roundtrip[s],
                    cp.sum([f[(j, i, s, c)] for (i, j) in leg_demand_arcs[s][leg]]) <= f_cap[c][s]*N_roundtrip[s] ]

  # Min flow
  for s in services:
    for (i, j) in flow_arcs[s]:
      for c in cargoes:
        constr += [f[(i, j, s, c)] >= q_min]

  # Port turnaround time
  for s in services:
    for (i, j) in legs[s]:
      constr += [  t_port[(i, j, s)] >= E_leg[(i, j, s)]/(P_cha_max*3600),
                   t_port[(i, j, s)] >= t_cargo ]

  # Availability
  for s in services:
    # Operating hours do not exceed ship availability
    constr += [ N_roundtrip[s]*cp.sum([(L_od[i,j]/v[s] + t_port[(i,j,s)]) for (i,j) in legs[s]]) <= t_avail[s]*N_vessel[s] ]

  # Demand
  # calculate sum of volume shipped by all vessels in all the o-d pairs
  # only enforce demand constraints for active o-d pairs
  for (i, j) in active_od:
    for c in cargoes:
      lhs = 0
      for s in services:
        if (i, j) in flow_arcs[s]:
          lhs += f[(i, j, s, c)]
      constr += [ lhs <= q_dem[c][i, j] ]

  # Utility (NOTE: scaling of the values by 10)
  flow_prod = 1
  for s in services:
    for (i, j) in flow_arcs[s]:
      for c in cargoes:
        flow_prod *= cp.power(10*f[(i, j, s, c)], U_bias[c]*L_od_norm[i, j])
  constr += [ flow_prod >= np.exp(0.99*U_min) ]    # NOTE, RHS MODIFIED

  #
  # Vessel design constraints
  #

  # Variable bounds
  for s in services:
    constr += [L_sup[s] >= 0.5*L[s], L_sup[s] <= 0.9*L[s]]
    constr += [p_td[s] >= p_min]

  for s in services:
    # Principal dimension bounds
    constr += [
        L[s] <= 240, L[s] >= 80,
        L[s]/(nabla[s]**(1/3)) >= 4.41,  L[s]/(nabla[s]**(1/3)) <= 1.2*7.27,
        L[s]/B[s] >= 3.96, L[s]/B[s] <= 7.13,
        B[s]/T[s] >= 2.31, B[s]/T[s]  <= 6.11
    ]

    leg_cons = []
    for (i, j)  in legs[s]:
      leg_cons.append(E_leg[(i, j , s)])

    # Frictional resistance
    # NOTE: cvxpy log is base e, which needs to be conerted to base 10
    constr += [r_Cf[s] + 2 <= cp.log( (v[s]/3.6)*L[s]/(1.004E-6) )/np.log(10) ]
    # Residual resistance
    constr += [ r_Cr_std[s]**1.69303 >= 258.6*(Fr[s]**4.83749) + 0.00929736*(Fr[s]**-1.44917) ]

    # Battery energy
    constr += [E_batt[s] >= cp.max(cp.vstack(leg_cons))*gamma_batt ]

    # Arc length integrations
    for i in range(4):  # enumerate the Simpson terms
      if i == 0:
        constr += [ r_g[i, s] >= 1,  r_g2[i, s] >= 1 ]
      else:
        constr += [ r_g[i, s] >= 1 + cp.power(2/B[s], 2*beta)*cp.power(T[s]*beta, 2)*cp.power(i*B[s]/6, 2*beta - 2),
                    r_g2[i, s] >= 1 + cp.power(2/B[s], 2*beta)*cp.power(T[s]*beta, 2)*cp.power(i*B_deck[s]/6, 2*beta - 2)]

    # Trasnverse stability
    constr += [ 0.4707*cp.power(B[s]/T[s], 1.4906) >= (h_KG[s] + eps_KG)/(beta_KB*T[s]) ]

    # Bending strength
    constr += [ k_safe*M_max[s]*90/(133*p_td[s]*B[s]*D[s]) <= sigma_y_max ]

    # Vertical force balance (buoyancy)
    constr += [nabla[s] >= W_sum[s]]

    # General arrangement
    for i in range(3):
      constr += [ w_batt[i, s]/2 <= (B[s]/2)*((2*lp_batt[i, s]/L[s])**0.5)*((hp_batt[s]/T[s])**(1/beta))]
      if i <= 1:
        constr += [ l_batt[i, s]*w_batt[i, s]*h_batt[s] == l_batt[i+1, s]*w_batt[i+1, s]*h_batt[s],   # set spaces to equal volume
                    l_batt[i+1, s] + lp_batt[i+1, s] <= lp_batt[i, s]]                                # relative space position

    constr += [ E_batt[s] <= 5*l_batt[0, s]*w_batt[0, s]*h_batt[s]*rho_vol,                           # total capacity of all battery spaces (5)
                h_batt[s] >= h_batt_min,                                                              # minimum battery space height
                hp_batt[s] >= 0.1*D[s],                                                               # minimum double bottom height
                l_CB >= (lp_batt[0, s] + l_batt[0, s]/2)/L[s],                                        # battery mass center located at the center of buouancy
                h_batt[s] + hp_batt[s] <= hp_roro[s],                                                 # first roro deck vertical position (distance from keel)
                hp_roro[s] + h_roro <= D[s],                                                          # strength deck vertical position (distance from keel)
                hp_roro[s] >= T[s] ]                                                                  # first roro deck position above waterline

  #
  # Objective
  #

  J_el = C_el_ann*cp.sum( [N_roundtrip[s]*E_leg[(i, j, s)] for s in services for (i,j) in legs[s]] )
  J_port = cp.sum( [(V_GT[s]/1000)*N_roundtrip[s]*C_port_ann for s in services for (i, j) in legs[s]] )
  J_crew = C_crew_ann*cp.sum([(f_cap[1][s]*N_crew_var + N_crew_const)*N_vessel[s] for s in services])
  J_batt = C_batt_ann*cp.sum([N_batt*k_batt*E_batt[s]*N_vessel[s] for s in services])
  J_ship = cp.sum([N_vessel[s]*(C_ship_ann[0]*cp.power(V_GT[s], C_ship_ann[1])) for s in services])

  objective_func = J_el + J_port + J_crew + J_batt + J_ship

  #
  # Problem construction
  #

  prob = cp.Problem(cp.Minimize(objective_func), constr)
  if solver=='ECOS':
    prob.solve(gp=True, verbose=True, solver='ECOS_BB', mi_rel_eps=1e-4)
  if solver=='MOSEK-OA':
    prob.solve(gp=True, verbose=True, solver='MOSEK', mosek_params={'MSK_DPAR_MIO_TOL_REL_GAP': 1e-4, 'MSK_IPAR_MIO_CONIC_OUTER_APPROXIMATION': 1})
  if solver=='MOSEK-BB':
    prob.solve(gp=True, verbose=True, solver='MOSEK', mosek_params={'MSK_DPAR_MIO_TOL_REL_GAP': 1e-4, 'MSK_IPAR_MIO_CONIC_OUTER_APPROXIMATION': 0})
  print('Status:', prob.status)
  print(f"Objective [kEUR]: {prob.value:.1f} ")

  #
  # Printout
  #

  for s in services:
    print(f"Service {s}")
    print(f"  N_roundtrip: {N_roundtrip[s].value:.2f}, N_vessel: {N_vessel[s].value:.2f}, E_batt [MWh]: {E_batt[s].value/3600:.2f}, GT: {V_GT[s].value:.1f}, L: {L[s].value:.2f}, L_sup: {L_sup[s].value:.2f}, B: {B[s].value:.2f}, T: {T[s].value:.2f}, D: {D[s].value:.2f}, p_td: {p_td[s].value:.4f}")
  print(J_el.value, J_port.value, J_crew.value, J_batt.value, J_ship.value)
  for s in services:
    print(f"v_{s}: ", v[s].value)

In [ ]:
####################################
# Linear problem implementation
####################################

def build_and_solve_linear(vessel_params, cost_params, network_params, solver='SCIP'):

  #
  # Data
  #

  speed_vec, E_max, R_tot, q_cap, V_GT, N_crew, A_sup, gamma_batt, eta_el, P_aux_unit \
  = map(vessel_params.get, ['speed_vec', 'E_max', 'R_tot', 'q_cap', 'V_GT', 'N_crew', 'A_sup', 'gamma_batt', 'eta_el', 'P_aux_unit'])

  N_batt, C_st_ann, C_batt_ann, C_crew_ann, C_el_ann, C_port_ann, C_ship_ann \
  = map(cost_params.get, ['N_batt', 'C_st_ann', 'C_batt_ann','C_crew_ann', 'C_el_ann', 'C_port_ann', 'C_ship_ann'])

  U_min, L_od, q_dem, routes, t_avail, N_trip, N_ship_max, t_cargo, U_bias, P_cha_max, q_min \
  = map(network_params.get, ['U_min', 'L_od', 'q_dem', 'routes', 't_avail',
                             'N_trip', 'N_ship_max', 't_cargo', 'U_bias', 'P_cha_max', 'q_min'])


  M_1 = 48
  k_batt = N_batt**-0.7

  N_ship_types = len(E_max)
  N_freqs = len(N_trip)
  N_speeds = len(speed_vec)

  #
  # Indexing sets
  #

  # Ship types
  ship_types = np.arange(N_ship_types)

  # Frequencies
  freqs = np.arange(N_freqs)

  # Speed levels
  speeds = np.arange(N_speeds)

  # Ports
  N_route_ports = { key:np.size(routes[key]) for key in routes.keys() }
  N_ports = max([max(v) for v in routes.values()])+1
  ports = np.arange(N_ports)

  # Routes / services
  N_services = len(routes)
  services = np.arange(N_services)

  # Cargoes
  cargoes = np.arange(2)

  # Sailing legs
  leg_demand_arcs = build_leg_flow_arcs(routes)
  flow_arcs = build_flow_arcs(leg_demand_arcs)
  legs = build_legs(routes)

  # Demand origin-destination pairs and sailing legs
  active_od = set()

  for s in services:
    active_od = active_od.union(flow_arcs[s])

  #
  # Decision variables
  #
  y = cp.Variable((N_services, N_ship_types), name='y', boolean=True)         # ship type selector
  N_ship = cp.Variable((N_services, N_ship_types), name="N_ship")             # num of ships in a service
  t_trip = cp.Variable((N_services), name='t_trip', nonneg=True)              # roundtrip duration, h

  z = {}         # 1 if frequency f is selected and 0 otherwise
  q = {}         # cargo quantity shipped, 1000 pax or 1000 lane-m
  q_loa = {}     # log approximation of cargo quantity shipped
  x = {}         # speed level, km/h
  t_cha = {}     # charging time, h
  t_port = {}    # port turnaround time, h
  t_hat_cha = {} # aux var for linearization
  t_hat_trip = {}

  for s in services:
    for (i, j) in flow_arcs[s]:
      for c in cargoes:
        q[(i, j, s, c)] = cp.Variable(name=f"q_{i}_{j}_{s}_{c}", nonneg=True)
        q_loa[(i, j, s, c)] = cp.Variable(name=f"q_la_{i}_{j}_{s}_{c}")

    for (i, j) in legs[s]:
      t_port[(i, j, s)] = cp.Variable(name=f"t_port_{i}_{j}_{s}", nonneg=True)
      t_cha[(i, j, s)] = cp.Variable(name=f"t_cha_{i}_{j}_{s}", nonneg=True)

    for v in speeds:
      for u in ship_types:
        x[(s, v, u)] = cp.Variable(name=f"t_cha_{s}_{v}_{u}", nonneg=True)

    for f in freqs:
      for u in ship_types:
        t_hat_trip[(s,f,u)] = cp.Variable(name=f"t_hat_trip_{s}_{f}_{u}", nonneg=True)
        t_hat_cha[(s,f,u)] = cp.Variable(name=f"t_hat_cha_{s}_{f}_{u}", nonneg=True)
        z[(s, f, u)] = cp.Variable(name=f"z_{s}_{f}_{u}", boolean=False, bounds=[0, 1])

  #
  # Derived quantities
  #

  # Leg-specific quantities
  E_leg = {}        # leg energy consumption, MJ (kN*km)
  t_transit = {}    # sailing time, h
  for (i, j) in active_od:
    for v in speeds:
      for u in ship_types:
        E_leg[(i, j, v, u)] = (R_tot[u, v]*L_od[i, j])/eta_el + 3.6*L_od[i, j]*P_aux_unit*A_sup[u]/speed_vec[v]     # MJ
      t_transit[(i, j, v)] = L_od[i, j]/speed_vec[v]      # h

  L_od_norm = L_od/np.max(L_od)                                       # normalized sailing distance

  #
  # Network constraints
  #

  constr = [ ]

  # Vessel capacity
  for s in services:
    for leg in leg_demand_arcs[s].keys(): # iterate legs (outbound) and impose a constr for each
      for c in cargoes:
        constr += [ cp.sum([q[(i, j, s, c)] for (i, j) in leg_demand_arcs[s][leg]]) <= cp.sum([q_cap[c, u]*N_trip[f]*z[(s,f,u)] for f in freqs for u in ship_types]),
                    cp.sum([q[(j, i, s, c)] for (i, j) in leg_demand_arcs[s][leg]]) <= cp.sum([q_cap[c, u]*N_trip[f]*z[(s,f,u)] for f in freqs for u in ship_types]) ]


  # Demand
  # calculate sum of volume shipped by all vessels in all the o-d pairs
  # only enforce demand constraints for active o-d pairs
  for (i, j) in active_od:
    for c in cargoes:
      lhs = 0
      for s in services:
        if (i, j) in flow_arcs[s]:
          lhs += q[(i, j, s, c)]
          constr += [q[(i, j, s, c)] >= q_min]
      constr += [ lhs <= q_dem[c][i, j] ]

  # Charging time (Note: E_leg is in MJ)
  for s in services:
    for (i, j) in legs[s]:
      constr += [ t_cha[(i, j, s)] == (1/(P_cha_max*3600))*cp.sum([E_leg[(i, j, v, u)]*x[(s, v, u)] for v in speeds for u in ship_types ]) ]

  # Port turnaround time
  for s in services:
    for (i, j) in legs[s]:
      constr += [ t_port[(i, j, s)] >= t_cha[(i, j, s)],
                 t_port[(i, j, s)] >= t_cargo]

  # Roundtrip voyage time
  for s in services:
    constr += [ t_trip[s] == cp.sum([ cp.sum([ t_transit[(i, j, v)]*x[(s, v, u)] for v in speeds for u in ship_types ]) + t_port[(i, j, s)] for (i,j) in legs[s]]) ]

  # Availability
  for s in services:
    constr += [ cp.sum([N_trip[f]*t_hat_trip[(s,f,u)] for u in ship_types for f in freqs]) <= t_avail[s]*cp.sum([N_ship[s, u] for u in ship_types]) ]


  # Battery capacity
  for s in services:
    for (i, j) in legs[s]:
      for u in ship_types:
        constr += [ cp.sum([E_leg[(i, j, v, u)]*x[(s, v, u)] for v in speeds])*gamma_batt <= E_max[u]*y[s, u] ]

  # Utility, log-app
  flow_sum = 0
  for s in services:
    for (i, j) in flow_arcs[s]:
      for c in cargoes:
        for k in [0.5, 1, 2, 4, 8, 16, 32, 64, 128, 256, 512]:
          khat = (k+0.5)/k
          constr += [ q_loa[(i, j, s, c)] <= 2*np.log(khat)*10*q[(i, j, s, c)] + np.log(k) + 2*k*np.log(1/khat) ]
        flow_sum += q_loa[(i, j, s, c)]*U_bias[c]*L_od_norm[i, j]
  constr += [ flow_sum >= U_min ]

  # Ship type selection logic
  for s in services:

    constr += [ cp.sum([y[s, u] for u in ship_types]) <= 1,                       # select one ship type per service
                cp.sum([z[(s, f, u)] for u in ship_types for f in freqs]) <= 1 ]  # select one freq per service and ship type

    for u in ship_types:

      constr += [ cp.sum([ z[(s, f, u)] for f in freqs ]) <= y[s, u],             # set freqs to zero unless ship type is selected
                  N_ship[s, u] <= N_ship_max*y[s, u] ]                            # num of ships is zero unless ship type is selected


      constr += [ cp.sum([ x[(s, v, u)] for v in speeds ]) == y[s, u] ]           # speed weights sum to for each service

  # Nonnegative and integer number of vessels
  for s in services:
    for u in ship_types:
      integer_N_ship = cp.FiniteSet(N_ship[s, u], np.array([0, 1, 2, 3, 4]))
      constr += [integer_N_ship]
      for f in freqs:
        binary_z = cp.FiniteSet(z[(s, f, u)], np.array([0,1]))
        constr += [binary_z]

  # Linear reformulations
  for s in services:
    for f in freqs:
      for u in ship_types:
        constr += [ t_hat_cha[(s,f,u)] >= cp.sum([t_cha[(i, j, s)] for (i,j) in legs[s]]) - M_1*(1-z[(s,f,u)]) ]
        constr += [ t_hat_trip[(s,f,u)] >= t_trip[s] - M_1*(1-z[(s,f,u)]) ]

  #
  # Objective
  #

  J_el = cp.sum([ N_trip[f]*C_el_ann*(P_cha_max*3600)*t_hat_cha[(s,f,u)] for s in services for u in ship_types for f in freqs ])
  J_port = cp.sum([ N_trip[f]*z[(s,f,u)]*C_port_ann*V_GT[u] for s in services for u in ship_types for (i,j) in legs[s] for f in freqs ])
  J_ship = cp.sum([ (C_ship_ann[0]*np.power(1000*V_GT[u], C_ship_ann[1]))*N_ship[s, u] for s in services for u in ship_types ])
  J_crew = cp.sum([ C_crew_ann*N_crew[u]*N_ship[s, u] for s in services for u in ship_types ])
  J_batt = cp.sum([C_batt_ann*N_batt*k_batt*E_max[u]*N_ship[s, u] for s in services for u in ship_types])

  objective_func = J_el + J_port + J_crew + J_batt + J_ship

  #
  # Problem construction
  #

  prob = cp.Problem(cp.Minimize(objective_func), constr)
  if solver == 'ECOS':
    prob.solve(verbose=True, solver='ECOS_BB')
  if solver == 'MOSEK':
    prob.solve(verbose=True, solver='MOSEK', mosek_params={'MSK_DPAR_MIO_TOL_REL_GAP': 1e-4})
  if solver == 'SCIP':
    prob.solve(verbose=True, solver='SCIP')
  print('Status:', prob.status)
  print(f"Objective [kEUR]: {prob.value:.1f} ")

  #
  # Printout
  #

  print(y.value)
  print(t_trip.value)
  print(N_ship.value)
  for s in services:
    for v in speeds:
      for u in ship_types:
        if x[(s, v, u)].value > 1E-4:
          print(f"x_{s}_{v}_{u}: ", x[(s, v, u)].value)

In [ ]:
####################################
# Linear model ship data
####################################

def generate_linear_vessel_data():
  params = {
      'speed_vec': [1, 20, 30, 40, 50],
      'E_max': 3600*np.array([310, 150, 100, 50, 310, 150, 100, 50, 300, 140, 90, 40, 300, 140, 90, 40, 290, 130, 80, 30, 130, 80, 30]),
      'R_tot': np.array([
                        [   1.9260448,   136.89357233,  290.03491751,  598.85102717, 1239.36413972],
                        [   1.9260448,   136.89357233,  290.03491751,  598.85102717, 1239.36413972],
                        [   1.9260448,   136.89357233,  290.03491751,  598.85102717, 1239.36413972],
                        [   1.9260448,   136.89357233,  290.03491751,  598.85102717, 1239.36413972],
                        [   1.9260448,   136.89357233,  290.03491751,  598.85102717, 1239.36413972],
                        [   1.9260448,   136.89357233,  290.03491751,  598.85102717, 1239.36413972],
                        [   1.9260448,   136.89357233,  290.03491751,  598.85102717, 1239.36413972],
                        [   1.9260448,   136.89357233,  290.03491751,  598.85102717, 1239.36413972],
                        [   1.40722579,  111.45564478,  245.37526818,  527.65325018, 1118.18599191],
                        [   1.40722579,  111.45564478,  245.37526818,  527.65325018, 1118.18599191],
                        [   1.40722579,  111.45564478,  245.37526818,  527.65325018, 1118.18599191],
                        [   1.40722579,  111.45564478,  245.37526818,  527.65325018, 1118.18599191],
                        [   1.40722579,  111.45564478,  245.37526818,  527.65325018, 1118.18599191],
                        [   1.40722579,  111.45564478,  245.37526818,  527.65325018, 1118.18599191],
                        [   1.40722579,  111.45564478,  245.37526818,  527.65325018, 1118.18599191],
                        [   1.40722579,  111.45564478,  245.37526818,  527.65325018, 1118.18599191],
                        [   1.05582394,  92.7784631,    214.747243,    485.931879,   1063.38589],
                        [   1.05582394,  92.7784631,    214.747243,    485.931879,   1063.38589],
                        [   1.01973453,  90.8187425,    209.893393,    472.825668,   1030.25329],
                        [   0.982994736, 88.8456566,    204.997120,    459.546741,   996.581837],
                        [   0.981231824, 88.7515552,    204.763358,    458.911231,   994.967680],
                        [   0.981231824, 88.7515552,    204.763358,    458.911231,   994.967680],
                        [   0.981231824, 88.7515552,    204.763358,    458.911231,   994.967680],
                        [   0.981231824, 88.7515552,    204.763358,    458.911231,   994.967680]]),
      'q_cap': np.array([[1.11, 1.11, 1.11, 1.11, 1.11, 1.11, 1.11, 1.11, 0.82, 0.82, 0.82, 0.82, 0.82, 0.82, 0.82, 0.82, 0.57, 0.57, 0.57, 0.57, 0.57, 0.57, 0.57, 0.57],
                        [2.41, 2.41, 2.41, 2.41, 1.34, 1.34, 1.34, 1.34, 1.77, 1.77, 1.77, 1.77, 0.98, 0.98, 0.98, 0.98, 1.23, 1.23, 1.23, 1.23, 0.68, 0.68, 0.68, 0.68]]),
      'V_GT': np.array([45.65, 45.65, 44.77, 44.42, 31.81, 31.81, 31.81, 31.81, 33.85, 33.85, 32.2, 31.1, 22.04, 22.04, 22.04, 22.04, 23.38, 23.38, 22.78, 22.18, 14.30, 14.30, 14.30, 14.30]),
      'N_crew': np.array([371, 371, 371, 371, 215, 215, 215, 215, 278, 278, 278, 278, 163, 163, 163, 163, 199, 199, 199, 199, 120, 120, 120, 120]),
      'A_sup': np.array([18, 18, 18, 18, 10, 10, 10, 10, 13.3, 13.3, 13.3, 13.3, 7.4, 7.4, 7.4, 7.4, 9.2, 9.2, 9.2, 9.2, 5.1, 5.1, 5.1, 5.1]),
      'gamma_batt': 1.5,
      'eta_el': 0.6,
      'P_aux_unit': 80,
    }

  return params

####################################
# Logcvx model ship data
####################################

def generate_logcvx_vessel_data():
  params = {
      'beta': 4,                  # hull fullness parameter
      'k_A': 0.95,                # hull wetted area correction factor
      'N_sup': 3,                 # number of decks in the superstructure
      'h_sup': 3.0,               # superstructure deck height, m
      'h_roro': 5.5,              # roro deck height, m
      'h_batt_min': 2.2,          # minimum battery space height, m
      'V_int_hat': 1.0E5,         # reference hull internal volume, m^3
      'k_vol': 0.8,               # hull volume block coefficient
      'eta_el': 0.6,              # total powertraing conversion efficiency (propeller and power electronics)
      'P_aux_unit': 0.08,         # auxiliary power, kW/m^2 accommodation floor area
      'A_sup_unit': 7.5,          # accommodation floor area per passenger, m^2 (roughly between 5 and 12)
      'beta_KB': 0.56195,         # relative vertical center of buoyancy, for beta = 6
      'eps_KG': 0.5,              # transverse stability margin
      'sigma_y_max': 235,         # maximum normal stress, MPa
      'k_safe': 2.0,              # girder stress limit safety factor
      'rho_vol': 167,             # battery space volumetric energy density, "MAERSK compact design", MJ/m^3
      'rho_mass': 0.417,          # battery space gravimetric energy density, "MAERSK compact design", t/m^3
      'rho_st': 8,                # steel density, t/m^3
      'p_sup': 0.01,              # thickness of non-girder plates (superstructure and transverse bulkheads), m
      'p_min': 0.012,             # minimum girder plate thickness
      'l_CB': 0.52394,            # relative longitudinal center of buoyancy for beta=6, m
      'gamma_batt': 1.5,          # battery reserve capacity factor
      'N_crew_const': 20,         # number of fixed crew members in deck and engineering
      'N_crew_var': 146,          # number of hotel crew members per 1k pax
    }

  return params

####################################
# Common cost data
####################################

def generate_cost_data():
  params = {
      'N_batt': 6,                # number of lifetime batteries
      'C_crew_ann': 38,           # annual cost of an onboard crew member, kEUR
      'C_st_ann': 0.99/20,        # annual steel cost, kEUR/t
      'C_batt_ann': 0.0447/20,    # annual battery system cost, 2030 estimate, kEUR/MJ (50 EUR/MJ = 180 EUR/kWh)
      'C_el_ann': 180*0.016667E-3,# annual electricity cost, kEUR/MJ
      'C_port_ann': 180*0.1,      # port charge, kEUR/(1000 GT)
      'C_ship_ann': (90/20, 0.7)  # annualized ship cost as a fuction of GT, kEUR (multiplier, exponent)
  }

  return params

In [ ]:
def generate_random_network(N_ports, N_services):
  # Seed
  np.random.seed(100)

  # Vector of port distances
  L_arc = np.random.randint(40, 200, size=(N_ports-1))

  # Matrix of sailing leg distances
  L_od = np.ones((N_ports, N_ports))
  for i in range(N_ports-1):
    dist = L_arc[i]
    for j in range(i+1,N_ports-1):
      L_od[i, j] = dist; L_od[j, i] = dist
      dist += L_arc[j]
    L_od[i,j+1] = dist; L_od[j+1,i] = dist

  # Demands
  q_dem = {0: 0.5 + 5*np.random.rand(N_ports, N_ports),
           1: 0.5 + 5*np.random.rand(N_ports, N_ports)}

  # Services
  routes = {0: np.arange(0, N_ports)}
  for i in range(1, N_services):
    N_service_ports = np.random.randint(2, N_ports-1)
    service_ports_unordered = np.random.choice(N_ports, N_service_ports, replace=False)
    service_ports = np.sort(service_ports_unordered)
    routes[i] = service_ports

  # Availability
  t_avail = 48*np.ones(N_services)

  # All params
  params = {
      'L_od': L_od,
      'q_dem': q_dem,
      'routes': routes,
      't_avail': t_avail,
      'N_trip': [1, 2, 4, 6, 8, 12, 16, 18],
      'N_ship_max': 4,
      't_cargo': 0.75,
      'U_bias': [1, 3],
      'P_cha_max': 60,
      'q_min': 0.05,
      'U_min': 10
  }
  return params

In [ ]:
####################################
# Run linear model
####################################

def run_lin1():
  vessel_params = generate_linear_vessel_data()
  cost_params = generate_cost_data()
  network_params = generate_random_network(4, 2)
  network_params.update({'U_min': 80})

  build_and_solve_linear(vessel_params, cost_params, network_params, solver='MOSEK')

run_lin1()

In [ ]:
####################################
# Run log-convex model
####################################

def run_logcvx1():
  vessel_params = generate_logcvx_vessel_data()
  cost_params = generate_cost_data()
  network_params = generate_random_network(4, 2)
  network_params.update({'U_min': 80})

  build_and_solve_logcvx(vessel_params, cost_params, network_params, solver='ECOS')

run_logcvx1()